# Lab 2 - Joint (optional, if there's time): global meets local

Only after **both** pull requests are merged. One screen, two people.

Global told you *what usually matters*; local told you *why one hour was high*. A **dependence plot**
bridges them: it shows, feature by feature, how the SHAP value changes as the feature changes.

In [ ]:
# --- setup: install SHAP, load the data, fit the model (just run this) ---
!pip install shap -q

import pandas as pd, numpy as np, matplotlib.pyplot as plt, shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

url = "https://raw.githubusercontent.com/drdave-teaching/opim5512-lab2-template/main/data/energy_model_data.csv"
df = pd.read_csv(url, parse_dates=["hour"])

FEATURES = ["temp_f", "hour_of_day", "dewpoint_f", "humidity_pct", "wind_kt", "weekend"]
X, y = df[FEATURES], df["load_mw"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"model R2 (test): {r2_score(yte, model.predict(Xte)):.2f}   |   typical miss: {mean_absolute_error(yte, model.predict(Xte)):,.0f} MW")
X.head()

In [ ]:
# --- SHAP setup (just run this): explain every prediction the model makes ---
explainer = shap.TreeExplainer(model)
shap_values = explainer(X)          # one row of SHAP values per hour, one column per feature
print("SHAP ready:", shap_values.shape, "(hours x features)")

Run a dependence plot for the top feature. `hour_of_day` is the interesting one - watch the
evening ramp. Saves `shap_dependence.png` for the report.

In [ ]:
shap.plots.scatter(shap_values[:, "hour_of_day"], color=shap_values[:, "temp_f"], show=False)
plt.gcf().savefig("shap_dependence.png", dpi=150, bbox_inches="tight"); plt.close()

from google.colab import files
files.download("shap_dependence.png")

### The Module 1 -> Module 2 payoff (write one sentence for the report)

In Lab 1 you found the hottest hour wasn't the peak-demand hour. Now the model *and* SHAP say why:
**hour-of-day carries as much weight as temperature.** The grid's evening rhythm is baked into demand,
not just the heat. Say that in plain English - it's the whole point of both labs.